## KNN Algorithm
K-Nearest Neighbour is a machine learning algorithm which can be used for both classification as well as regression problems.

## Classification with KNN
KNN is one of the most simplest classifier. In this algorithm :
1. Assumption: Points belonging to the same class tend to be close to one another in the feature space.
2. We have a new query point whose class is unknown.
3. Find the points in the training data that are closest to it.
4. Look at their class labels.
5. Assign the query point the class that is most common among those neighbors.

<div style="text-align: center;">
  <img src="./assets/KNN.png" width="700px">
</div>

In this algorithm $K$ value is the number of nearest points, which are chosen for the classification of the query point. If $K=3$, we look at the 3 nearest points to the query point and look at their class. If 2 are of class B and 1 of class A we classify the query point as class B. If we take even no. of $K$ and equal no. of classes exist we can added some parameters to make the classification easier like :
- look at the distance of the tied neighbors and give more weight to closer ones,
- choose one class according to some fixed rule,
- or use a different method for breaking ties.  

### Finding the nearest points 
To find n no. of nearest points to the query point we have two ways :
1. Brute Force Approach : We can find individual distances of all points to the query point and choose the n smallest distances. Time complexity is O(N).
2. K-D Tree : We can use K-D Tree data structure to find the n nearest nearest points to query point.

In [1]:
# Brute For Approach
import math

# Training data
X = [
    [1, 1],
    [2, 1],
    [1, 2],
    [8, 8],
    [9, 8],
    [8, 9]
]

# Corresponding class labels
y = ["A", "A", "A", "B", "B", "B"]


# Query point
query = [3, 2]

# Number of neighbors
K = 3


# Step 1: Calculate distance from query to EVERY point
distances = []

for i in range(len(X)):
    distance = math.sqrt(
        (query[0] - X[i][0])**2 +
        (query[1] - X[i][1])**2
    )

    distances.append((distance, y[i]))


# Step 2: Sort by distance
distances.sort()


# Step 3: Take K nearest neighbors
neighbors = distances[:K]


# Step 4: Majority vote
votes = {}

for distance, label in neighbors:
    votes[label] = votes.get(label, 0) + 1


# Step 5: Class with most votes
prediction = max(votes, key=votes.get)

print("Nearest neighbors:", neighbors)
print("Prediction:", prediction)

Nearest neighbors: [(1.4142135623730951, 'A'), (2.0, 'A'), (2.23606797749979, 'A')]
Prediction: A


In [2]:
# K-D Tree Approach
import math

# -----------------------------
# Training data
# -----------------------------

X = [
    [1, 1],
    [2, 1],
    [1, 2],
    [8, 8],
    [9, 8],
    [8, 9]
]

y = ["A", "A", "A", "B", "B", "B"]


# -----------------------------
# K-D Tree Node
# -----------------------------

class KDNode:
    def __init__(self, point, label, axis):
        self.point = point
        self.label = label
        self.axis = axis

        self.left = None
        self.right = None


# -----------------------------
# Build K-D Tree
# -----------------------------

def build_kdtree(points, labels, depth=0):

    if not points:
        return None

    # Which dimension should we split on?
    axis = depth % len(points[0])

    # Sort points according to current dimension
    data = sorted(zip(points, labels),
                  key=lambda x: x[0][axis])

    # Choose median point
    median = len(data) // 2

    point, label = data[median]

    node = KDNode(point, label, axis)

    # Points smaller than median
    left_points = [p for p, l in data[:median]]
    left_labels = [l for p, l in data[:median]]

    # Points larger than median
    right_points = [p for p, l in data[median + 1:]]
    right_labels = [l for p, l in data[median + 1:]]

    node.left = build_kdtree(left_points, left_labels, depth + 1)
    node.right = build_kdtree(right_points, right_labels, depth + 1)

    return node


# -----------------------------
# Distance function
# -----------------------------

def distance(p1, p2):

    return math.sqrt(
        sum((a - b) ** 2 for a, b in zip(p1, p2))
    )


# -----------------------------
# K nearest neighbors
# -----------------------------

def knn_search(node, query, K, best=None):

    if node is None:
        return best

    if best is None:
        best = []

    # Distance from query to current node
    dist = distance(query, node.point)

    # Store this point
    best.append((dist, node.point, node.label))

    # Keep only K closest points
    best.sort(key=lambda x: x[0])
    best = best[:K]

    # Determine which side to search first
    axis = node.axis

    if query[axis] < node.point[axis]:
        near = node.left
        far = node.right
    else:
        near = node.right
        far = node.left

    # Search the closer side first
    best = knn_search(near, query, K, best)

    # Distance from query to splitting plane
    plane_distance = abs(query[axis] - node.point[axis])

    # If there could be a closer point on the other side,
    # search that side too.
    if len(best) < K or plane_distance < best[-1][0]:
        best = knn_search(far, query, K, best)

    return best


# -----------------------------
# Build tree
# -----------------------------

tree = build_kdtree(X, y)


# -----------------------------
# Query
# -----------------------------

query = [3, 2]
K = 3

neighbors = knn_search(tree, query, K)

print("Nearest neighbors:")

for dist, point, label in neighbors:
    print(
        "Point:", point,
        "Class:", label,
        "Distance:", round(dist, 2)
    )


# -----------------------------
# Majority vote
# -----------------------------

votes = {}

for dist, point, label in neighbors:
    votes[label] = votes.get(label, 0) + 1


prediction = max(votes, key=votes.get)

print("\nPrediction:", prediction)

Nearest neighbors:
Point: [2, 1] Class: A Distance: 1.41
Point: [1, 2] Class: A Distance: 2.0
Point: [1, 1] Class: A Distance: 2.24

Prediction: A
